# Feature Engineering Phase – Building Predictive Intelligence

Feature engineering is one of the most critical stages in a machine learning pipeline, as it directly determines how well the model can understand customer behavior and identify churn patterns. In this phase, raw telecom customer data is transformed into meaningful, structured, and predictive features that capture customer behavior, financial patterns, service usage, and risk signals.

Unlike raw data, engineered features provide deeper insight into customer lifecycle patterns, enabling the model to detect hidden relationships that are not explicitly visible in the original dataset.

##  Objectives of this Phase

- Transform raw customer data into meaningful predictive features  
- Capture customer behavior, engagement, and financial risk patterns  
- Improve model performance through better data representation  
- Create business-relevant features for churn interpretation  
- Prepare dataset for machine learning modeling and SHAP explainability 

In [1]:
import sys
import os

PROJECT_ROOT = r"c:\Projects\TEYZIX-CORE-INTERNSHIP\task-3_Telco_Churn_System"
sys.path.append(PROJECT_ROOT)

In [2]:
import pandas as pd

In [3]:
data=pd.read_csv(r"../dataset/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [4]:
data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
data = data.drop("customerID", axis=1)

In [6]:
data["Churn"] = data["Churn"].map({"Yes": 1, "No": 0})

In [7]:
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
data["TotalCharges"].fillna(data["TotalCharges"].median(), inplace=True)

C:\Users\10\AppData\Local\Temp\ipykernel_14260\545638472.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data["TotalCharges"].fillna(data["TotalCharges"].median(), inplace=True)


## Monthly Risk Ratio (Customer Value Feature)

In [8]:
data["AvgMonthlySpend"] = data["TotalCharges"] / (data["tenure"] + 1)

## Tenure Groups (Customer Lifecycle)

In [9]:
data["TenureGroup"] = pd.cut(
    data["tenure"],
    bins=[0, 12, 24, 48, 72],
    labels=["New", "ShortTerm", "MidTerm", "LongTerm"]
)

## Service Count Feature:

In [10]:
service_cols = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies"
]

data["ServiceCount"] = (data[service_cols] == "Yes").sum(axis=1)

## Contract Risk Feature

In [11]:
data["ContractRisk"] = data["Contract"].map({
    "Month-to-month": 2,
    "One year": 1,
    "Two year": 0
})

##  Payment Risk Feature

In [12]:
data["IsElectronicCheck"] = (data["PaymentMethod"] == "Electronic check").astype(int)

##  Family Support Feature

In [13]:
data["HasFamily"] = ((data["Partner"] == "Yes") | (data["Dependents"] == "Yes")).astype(int)

##  High Value Customer Flag

In [14]:

data["HighValue"] = (data["MonthlyCharges"] > data["MonthlyCharges"].median()).astype(int)

## HANDLE CATEGORICAL VARIABLES

In [15]:
data = pd.get_dummies(data, drop_first=True)

In [16]:
data.shape

(7043, 40)

In [17]:
data.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,AvgMonthlySpend,ServiceCount,ContractRisk,IsElectronicCheck,HasFamily,...,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,TenureGroup_ShortTerm,TenureGroup_MidTerm,TenureGroup_LongTerm
0,0,1,29.85,29.85,0,14.925000,1,2,1,1,...,False,False,False,True,False,True,False,False,False,False
1,0,34,56.95,1889.50,0,53.985714,2,1,0,0,...,False,True,False,False,False,False,True,False,True,False
2,0,2,53.85,108.15,1,36.050000,2,2,0,0,...,False,False,False,True,False,False,True,False,False,False
3,0,45,42.30,1840.75,0,40.016304,3,1,0,0,...,False,True,False,False,False,False,False,False,True,False
4,0,2,70.70,151.65,1,50.550000,0,2,1,0,...,False,False,False,True,False,True,False,False,False,False
